# NDMC Conductivity MPC Notebook

This notebook is a reproducible NDMC conductivity MPC experiment.

It does four things:
- rerun one NDMC closed-loop case built on `build_tracking_mpc(...)`;
- read the saved CSV outputs;
- visualize the closed-loop behavior in a publication-style format;
- inspect timing tables when benchmarking is needed.


In [ ]:
using Pkg

bootstrap_path = joinpath(pwd(), "notebooks", "bootstrap_examples_notebook_env.jl")
if !isfile(bootstrap_path)
    bootstrap_path = joinpath(pwd(), "bootstrap_examples_notebook_env.jl")
end
include(bootstrap_path)

# Always instantiate here so a fresh machine installs the notebook dependencies
# before the first `using CSV, DataFrames, Plots` call.
env_info = prepare_examples_notebook_env(dirname(bootstrap_path); instantiate=true, io=devnull)
repo_root = env_info.repo_root
examples_env = env_info.examples_env
generated_dir = env_info.generated_dir
ndmc_module_path = env_info.ndmc_module_path

using EOptInterface
using CSV, DataFrames, Plots
using Statistics

include(ndmc_module_path)
using .NDMCExample

apply_eoi_publication_style!()


## Re-run the NDMC Example

This cell runs the same NDMC case used by the script version.
The plant equations, MPC setup, and closed-loop simulation are kept in `examples/ndmc_case.jl` so the experiment can be read directly.

By default it enables the NDMC-specific detailed live log so each MPC trigger prints one status line in the cell output.
You can also enable the package's generic `show_status` line from the rerun cell below if you want both views at once.

The default case uses:
- `dt = 20 s`;
- `P = 20`;
- `M = 3`;
- one low-zone influent shock from `2100 s` to `2250 s`;
- one shared `Q_air` manipulated variable.

The figures below use the same disturbance window as the paper-style benchmark discussion.


In [ ]:
rerun_closed_loop = false
rerun_show_detailed_status = true
rerun_show_generic_status = false

rerun_result = nothing

if rerun_closed_loop
    rerun_cfg = NDMCMPCConfig(
        show_detailed_status = rerun_show_detailed_status,
        show_generic_status = rerun_show_generic_status,
    )
    rerun_result = run_ndmc_case(
        rerun_cfg;
        save_outputs = true,
        announce_outputs = true,
        output_dir = generated_dir,
    )
end


## Optional Time Profile of NDMC MPC
Set `run_workflow_profile = true` only when timing data are needed. This reruns the case several times to separate compilation, warm execution, callbacks, and simulation.


In [ ]:
run_workflow_profile = false
profiled_show_detailed_status = false
profiled_show_generic_status = false
profile_report = nothing

if run_workflow_profile
    profile_cfg = NDMCMPCConfig(
        show_detailed_status = profiled_show_detailed_status,
        show_generic_status = profiled_show_generic_status,
    )
    profile_report = NDMCExample.run_ndmc_workflow_timing_report(
        profile_cfg;
        output_dir = generated_dir,
        save_outputs = true,
        announce_outputs = true,
    )
    for table in (
        profile_report.summary_view,
        profile_report.runtime_split_view,
        profile_report.callback_breakdown_view,
        profile_report.comparison_view,
        profile_report.budget_view,
        profile_report.online_step_stats,
        profile_report.env_view,
    )
        display(table)
    end
    display(profile_report.profile_dashboard)
else
    println("Workflow profiling skipped. Set run_workflow_profile = true to run it.")
end


## Optional Isolated BenchmarkTools Timing
Set `run_isolated_btime = true` to time individual warm stages outside the closed-loop sequence.


In [ ]:
run_isolated_btime = false
btime_report = nothing
btime_snapshot_time = NDMCExample.ndmc_default_btime_snapshot_time(NDMCMPCConfig())
btime_samples = 10
btime_evals = 1
btime_seconds = 2.0

if run_isolated_btime
    btime_cfg = NDMCMPCConfig(
        show_detailed_status = false,
        show_generic_status = false,
    )
    btime_report = NDMCExample.run_ndmc_isolated_btime_report(
        btime_cfg;
        snapshot_time = btime_snapshot_time,
        samples = btime_samples,
        evals = btime_evals,
        seconds = btime_seconds,
        output_dir = generated_dir,
        workflow_report = profile_report,
        save_outputs = true,
        save_workbook = true,
        announce_outputs = true,
    )
    display(btime_report.summary)
    display(btime_report.snapshot)
else
    println("Isolated BenchmarkTools timing skipped. Set run_isolated_btime = true to run it.")
end


## Optional Whole-Workflow BenchmarkTools Timing

Set `run_whole_workflow_btime = true` to benchmark the complete warm NDMC workflow as one target.


In [ ]:
run_whole_workflow_btime = false
whole_workflow_summary = nothing
whole_workflow_samples = 3
whole_workflow_evals = 1
whole_workflow_seconds = 30.0

if run_whole_workflow_btime
    whole_workflow_cfg = NDMCMPCConfig(
        show_detailed_status = false,
        show_generic_status = false,
    )
    whole_workflow_summary = NDMCExample.run_ndmc_whole_workflow_btime(
        whole_workflow_cfg;
        samples = whole_workflow_samples,
        evals = whole_workflow_evals,
        seconds = whole_workflow_seconds,
    )
    display(whole_workflow_summary)
else
    println("Whole-workflow timing skipped. Set run_whole_workflow_btime = true to run it.")
end


In [ ]:
closed_loop_path = joinpath(generated_dir, "ndmc_conductivity_eopt_closed_loop.csv")
applied_path = joinpath(generated_dir, "ndmc_conductivity_eopt_applied_control.csv")
name_map_path = joinpath(generated_dir, "ndmc_conductivity_eopt_name_map.csv")

isfile(closed_loop_path) || error("Missing $(closed_loop_path). Run the example cell first.")
isfile(applied_path) || error("Missing $(applied_path). Run the example cell first.")
isfile(name_map_path) || error("Missing $(name_map_path). Run the example cell first.")

closed_loop = CSV.read(closed_loop_path, DataFrame)
applied = CSV.read(applied_path, DataFrame)
name_map = CSV.read(name_map_path, DataFrame)

first(closed_loop, 5)

In [ ]:
ndmc_plot_penalty_terms(applied; disturbance_window = NDMC_NOTEBOOK_PLOT_CONFIG.disturbance_window)


In [ ]:
ndmc_plot_penalty_terms(applied)


## Names Used by the NDMC Example

This example does not hand-build JuMP names.
Instead, it passes `base_name` through `MPCControlSpec` and `MPCOutputSpec`, then reuses the naming table stored with the MPC problem.

That matters for two reasons:
- the example stays cleaner;
- downstream code can still find the right trajectory by a human-readable name.


In [ ]:
name_map


## Compare with the Legacy DMC Baseline

This section compares the current EOptInterface MPC result with the legacy DMC implementation in `examples/MPC_NDMC.jl`.

It keeps the same low-zone shock benchmark and uses the same paper-style disturbance window for the focused comparison plots.
The comparison data are stored in `examples/generated`, and the cell below can refresh them if the current MPC result or the legacy script changes.


In [ ]:
rerun_legacy_comparison = false

legacy_script = joinpath(repo_root, "examples", "MPC_NDMC.jl")
legacy_generator = joinpath(repo_root, "examples", "generate_ndmc_legacy_comparison.jl")
legacy_summary_path = joinpath(generated_dir, "ndmc_vs_legacy_summary_10s.csv")
legacy_path = joinpath(generated_dir, "legacy_mpc_ndmc_closed_loop_10s.csv")
comparison_path = joinpath(generated_dir, "ndmc_vs_legacy_comparison_10s.csv")

comparison_outputs = [legacy_summary_path, legacy_path, comparison_path]
comparison_inputs = [closed_loop_path, legacy_script, legacy_generator]

comparison_outputs_ready = all(isfile, comparison_outputs)
comparison_outputs_stale = !comparison_outputs_ready || maximum(stat(path).mtime for path in comparison_inputs) > minimum(stat(path).mtime for path in comparison_outputs)
rerun_legacy_comparison = rerun_legacy_comparison || comparison_outputs_stale

if rerun_legacy_comparison
    cmd = `$(Base.julia_cmd()) --project=$(examples_env) $legacy_generator $closed_loop_path $legacy_path $comparison_path $legacy_summary_path`
    run(cmd)
end


In [ ]:
legacy_summary = CSV.read(legacy_summary_path, DataFrame)
legacy_summary


In [ ]:
legacy_closed_loop = CSV.read(legacy_path, DataFrame)

shock_window = NDMC_PLOT_CONFIG.shock_window
disturbance_window = NDMC_PLOT_CONFIG.disturbance_window

paper_pair = ndmc_plot_shock_window_pair(
    closed_loop,
    legacy_closed_loop;
    tspan = shock_window,
    disturbance_window = disturbance_window,
)
full_pair = ndmc_plot_legacy_comparison_pair(
    closed_loop,
    legacy_closed_loop;
    disturbance_window = disturbance_window,
    title_suffix = "full horizon",
)

display(paper_pair)
full_pair


In [ ]:
comparison_df = CSV.read(comparison_path, DataFrame)
ndmc_plot_difference_pair(comparison_df; disturbance_window = NDMC_PLOT_CONFIG.disturbance_window)
